In [1]:
import pandas as pd
import numpy as np
import joblib

from datetime import datetime

In [2]:
test = pd.read_csv("test.csv")

test_bkup = test.drop(columns=["client", "close", "price_am", "price_pm"])

In [3]:
# datetime列をdatetime64型に変換
test["datetime"] = pd.to_datetime(test["datetime"])

# year, month, day, weekday列を作成
test["year"] = test["datetime"].dt.year
test["month"] = test["datetime"].dt.month
test["day"] = test["datetime"].dt.day
test["weekday"] = test["datetime"].dt.dayofweek

# 不要なdatetime列を削除
test.drop("datetime", axis=1, inplace=True)

test.head()

,client,close,price_am,price_pm,year,month,day,weekday
0,1,0,3,2,2016,4,1,4
1,0,0,5,5,2016,4,2,5
2,1,0,2,2,2016,4,3,6
3,1,0,1,1,2016,4,4,0
4,0,0,1,1,2016,4,5,1


In [4]:
#休日フラグがオンなら無条件で0にする
test.loc[test["close"] == 1, ["price_am", "price_pm"]] = 0

In [5]:
#clientが1とそれ以外に分割
client_test = test.query("client == 1")
test = test.query("client == 0")

In [6]:
# 保存されたモデルをロードして予測を行う関数を作成
def predict_with_models(models, X):
    preds = np.zeros((len(models), X.shape[0]))
    for i, model in enumerate(models):
        preds[i, :] = model.predict(X)
    return preds.mean(axis=0)

In [7]:
# モデルをロード
client_loaded_models = joblib.load("Apple_client_model.pkl")
loaded_models = joblib.load("Apple_model.pkl")

In [8]:
#予測
client_predictions = predict_with_models(client_loaded_models, client_test)
predictions = predict_with_models(loaded_models, test)

In [9]:
client_test.loc[:, "y"] = client_predictions

test.loc[:, "y"] = predictions

In [10]:
#休日フラグがオンなら無条件で0にする
client_test.loc[client_test["close"] == 1, ["y"]] = 0
test.loc[test["close"] == 1, ["y"]] = 0

In [11]:
inference_client = client_test.drop(columns=["client", "close", "price_am", "price_pm", "year", "month", "day",  "weekday"])

inference = test.drop(columns=["client", "close", "price_am", "price_pm", "year", "month", "day",  "weekday"])

In [12]:
inference_client.head()

,y
0,75.550690
2,70.640538
3,61.032520
5,58.241189
6,57.477827


In [13]:
inference.head()

,y
1,63.243964
4,47.227229
7,42.205765
16,40.707560
17,36.917893


In [14]:
inference = pd.concat([test_bkup, inference_client, inference])

inference = inference.sort_index()

inference = inference.groupby(inference.index).first()

inference.head()

,datetime,y
0,2016-04-01,75.550690
1,2016-04-02,63.243964
2,2016-04-03,70.640538
3,2016-04-04,61.032520
4,2016-04-05,47.227229


In [15]:
inference.to_csv("inference.csv", index=False, header=None)